# How this becomes a paper — the scaled-up version
### Fifteen brain disorders x two parcellations x spatial nulls

This is the "second project" the source document describes in Part 6, built out as an actual notebook
rather than left as guidance. It is a separate, larger piece of work from
`alzheimers_selective_vulnerability.ipynb` (the single-disease "weekend project"), and reuses that
notebook's `DATA_DIR` on Google Drive so the ~4GB Allen Human Brain Atlas download isn't repeated if you've
already run it there.

**Run this in Google Colab.** Every external download this notebook needs — the AHBA backend
(`api.brain-map.org`), the GWAS Catalog (`www.ebi.ac.uk`) — was confirmed blocked by this session's sandbox
network policy; only Colab's open internet access makes this runnable.

**What's implemented vs. what isn't**, against the document's own Part 6 checklist:

| Document asks for | Status |
|---|---|
| 6.2: gene lists for ~15 disorders, region-scored | Implemented — 15 disorders below |
| 6.3: cluster diseases by regional pattern | Implemented |
| 6.3: do psychiatric/neurodegenerative separate | Implemented (via the clustering above) |
| 6.3: how many published associations survive spatial-null correction | **Not implemented** — see the note in the "Literature replication" section; it needs a curated ground-truth table of previously-published disease-region findings, which I won't fabricate |
| 6.4: redo with a second parcellation | Implemented — Schaefer-200, via a verified cross-atlas spatial-overlap mapping (see below), not just a second run in isolation |
| 6.4: spatial nulls throughout | Implemented for the target-region checks; **not** extended to the clustering step (a multivariate spin test for comparing *patterns between diseases* is a further step beyond what the document specifies, and beyond what I've verified here — flagged, not silently skipped) |
| 6.4: deal with gene list size | Implemented — explicit diagnostic, not a hidden correction |
| 6.4: state the nearest-gene problem plainly | Stated below |

**Verified before writing this notebook, not assumed:** the real Schaefer-2018 atlas was downloaded in this
session (`nilearn.datasets.fetch_atlas_schaefer_2018` — reachable via `raw.githubusercontent.com`, unlike
the AHBA/GWAS/neuromaps-surface hosts) and its actual label format and value range confirmed. The
DK-entorhinal-cortex-to-Schaefer-parcel spatial overlap below was computed for real, on the real downloaded
atlases, in this session — not estimated: 1,904 of 1,953 DK left-entorhinal voxels (97.5%) fall inside a
single Schaefer-100 parcel, `7Networks_LH_Limbic_TempPole_1`. The disease list, gene-list-size diagnostic,
and clustering logic were smoke-tested against synthetic data shaped like the real multi-disease output.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DATA_DIR = '/content/drive/MyDrive/ahba_alzheimers'  # same directory as the single-disease notebook
SCALE_DIR = f'{DATA_DIR}/scale_up'
GWAS_DIR = f'{DATA_DIR}/gwas'
os.makedirs(SCALE_DIR, exist_ok=True)
os.makedirs(GWAS_DIR, exist_ok=True)

In [ ]:
!pip install -q abagen neuromaps nilearn statsmodels nibabel scipy scikit-learn

## 1. The disease list

Fifteen disorders, six neurodegenerative and nine psychiatric/other, per the document's "neurodegenerative
ones like Alzheimer's and Parkinson's, psychiatric ones like schizophrenia and bipolar disorder."

`trait_keyword` is matched as a case-insensitive substring against the GWAS Catalog's `DISEASE/TRAIT`
column (Part 3 below) — **inspect the matched trait strings before trusting them**; this notebook does not
assert a specific EFO code for any of these fifteen, precisely because that would be fifteen unverified
claims instead of one (the single-disease notebook already flagged this problem for just Alzheimer's).

`dk_target` is a Desikan-Killiany region-label substring for disorders with a well-established single
anatomical target in classical neuroanatomy. It's `None` where there isn't one — including cases where the
textbook target region exists but isn't resolvable in this atlas at all (Parkinson's substantia nigra and
essential tremor's cerebellum are both absent from Desikan-Killiany's 83 regions), and cases that are
genuinely diffuse or circuit-based rather than focal (schizophrenia, per the source document's own framing
in Part 4.3). Don't read `None` as "this notebook forgot" — it's a claim about the anatomy, stated
explicitly rather than papered over with a guess.

In [ ]:
import pandas as pd

DISEASES = pd.DataFrame([
    # name,                          trait_keyword,               category,           dk_target,             note
    ("Alzheimer's disease",          "Alzheimer",                  "neurodegenerative", "entorhinal|hippocampus", None),
    ("Parkinson's disease",          "Parkinson",                  "neurodegenerative", None, "target (substantia nigra) not resolvable in Desikan-Killiany"),
    ("Huntington's disease",         "Huntington",                 "neurodegenerative", "caudate|putamen", "target is purely subcortical -> no Schaefer equivalent exists (verified, see Section 4)"),
    ("Amyotrophic lateral sclerosis","amyotrophic lateral sclerosis", "neurodegenerative", "precentral", "primary motor cortex"),
    ("Frontotemporal dementia",      "frontotemporal dementia",    "neurodegenerative", None, "diffuse across frontal+temporal, no single clean parcel"),
    ("Multiple sclerosis",           "multiple sclerosis",         "neurodegenerative", None, "primarily white-matter disease; a grey-matter atlas is a poor fit"),
    ("Schizophrenia",                "Schizophrenia",              "psychiatric",       None, "no single clean anatomical target — see source document Part 4.3"),
    ("Bipolar disorder",             "Bipolar disorder",           "psychiatric",       None, None),
    ("Major depressive disorder",    "Depression",                 "psychiatric",       None, None),
    ("Autism spectrum disorder",     "Autism",                     "psychiatric",       None, None),
    ("ADHD",                         "Attention deficit",          "psychiatric",       None, None),
    ("Obsessive-compulsive disorder","Obsessive-compulsive",       "psychiatric",       "lateralorbitofrontal", "tentative — orbitofrontal-striatal circuit, weaker anatomical consensus than Alzheimer's"),
    ("Epilepsy",                     "Epilepsy",                   "other",             None, "heterogeneous focus by epilepsy subtype"),
    ("Migraine",                     "Migraine",                   "other",             None, None),
    ("Essential tremor",             "Essential tremor",           "other",             None, "target (cerebellum) not resolvable in Desikan-Killiany"),
], columns=["name", "trait_keyword", "category", "dk_target", "note"])

DISEASES

## 2. Atlas 1: Desikan-Killiany expression data (reuse)

If you've already run the single-disease notebook against this same `DATA_DIR`, this loads the cached
`ahba.csv` instead of re-downloading ~4GB.

In [ ]:
import abagen

atlas_dk = abagen.fetch_desikan_killiany()
info_dk = pd.read_csv(atlas_dk['info'])

HEMISPHERE_MODE = "left_only"  # keep consistent with the single-disease notebook

ahba_cache = f'{DATA_DIR}/ahba.csv'
if os.path.exists(ahba_cache):
    expression_dk = pd.read_csv(ahba_cache, index_col=0)
    expression_dk.columns = expression_dk.columns.astype(str)
    print(f"Loaded cached expression data: {expression_dk.shape}")
else:
    if HEMISPHERE_MODE == "left_only":
        expression_dk = abagen.get_expression_data(atlas_dk['image'], atlas_dk['info'], lr_mirror=None, data_dir=DATA_DIR)
        keep_ids = info_dk.loc[info_dk['hemisphere'].isin(['L', 'B']), 'id']
        expression_dk = expression_dk.loc[expression_dk.index.isin(keep_ids)]
    else:
        expression_dk = abagen.get_expression_data(atlas_dk['image'], atlas_dk['info'], lr_mirror='bidirectional', data_dir=DATA_DIR)
    expression_dk.to_csv(ahba_cache)
    print(f"Downloaded and cached expression data: {expression_dk.shape}")

## 3. Atlas 2: Schaefer-200 — the second parcellation

Confirmed working in this session: `nilearn.datasets.fetch_atlas_schaefer_2018` downloads from
`raw.githubusercontent.com`, which (unlike the AHBA and GWAS Catalog hosts) is reachable even from this
sandbox — so the atlas fetch below has actually been executed and verified, not just written from the
docs. 200 regions (100 per hemisphere, 7 Yeo networks) — a similar order of magnitude to
Desikan-Killiany's 68 cortical regions, but a genuinely different parcellation scheme, which is the point.

Schaefer is cortex-only (no subcortical/brainstem regions), and its labels carry no classical anatomical
names — they're Yeo-network-based (e.g. `7Networks_LH_Limbic_TempPole_1`), confirmed against the real
downloaded label file. Hemisphere is parsed from the `LH`/`RH` substring in each label, which is
verified present in the real labels, not assumed.

In [ ]:
from nilearn import datasets

schaefer = datasets.fetch_atlas_schaefer_2018(n_rois=200, yeo_networks=7, data_dir=f'{SCALE_DIR}/schaefer')

sch_labels = [l.decode() if isinstance(l, bytes) else l for l in schaefer.labels]
sch_labels = [l for l in sch_labels if l != 'Background']
assert len(sch_labels) == 200, f"expected 200 labels, got {len(sch_labels)} — Schaefer release may have changed"

info_schaefer = pd.DataFrame({'id': range(1, len(sch_labels) + 1), 'label': sch_labels})
info_schaefer['hemisphere'] = info_schaefer['label'].apply(
    lambda l: 'L' if 'LH' in l else ('R' if 'RH' in l else '?')
)
info_schaefer['structure'] = 'cortex'
assert (info_schaefer['hemisphere'] != '?').all(), "some Schaefer labels didn't parse to L or R — inspect info_schaefer"
info_schaefer.to_csv(f'{SCALE_DIR}/schaefer_info.csv', index=False)
info_schaefer.head()

In [ ]:
schaefer_cache = f'{SCALE_DIR}/ahba_schaefer.csv'
if os.path.exists(schaefer_cache):
    expression_schaefer = pd.read_csv(schaefer_cache, index_col=0)
    expression_schaefer.columns = expression_schaefer.columns.astype(str)
    print(f"Loaded cached Schaefer expression data: {expression_schaefer.shape}")
else:
    if HEMISPHERE_MODE == "left_only":
        expression_schaefer = abagen.get_expression_data(
            schaefer.maps, info_schaefer, lr_mirror=None, data_dir=DATA_DIR
        )
        keep_ids = info_schaefer.loc[info_schaefer['hemisphere'] == 'L', 'id']
        expression_schaefer = expression_schaefer.loc[expression_schaefer.index.isin(keep_ids)]
    else:
        expression_schaefer = abagen.get_expression_data(
            schaefer.maps, info_schaefer, lr_mirror='bidirectional', data_dir=DATA_DIR
        )
    expression_schaefer.to_csv(schaefer_cache)
    print(f"Downloaded and cached Schaefer expression data: {expression_schaefer.shape}")

## 4. Cross-atlas target mapping — verified, not estimated

To check a disease's known target region under *both* parcellations, the DK target label needs a Schaefer
equivalent. Rather than guessing one, this computes it: resample the Schaefer volumetric atlas onto the DK
image's grid (nearest-neighbour, appropriate for label images) and find which Schaefer parcel(s) actually
occupy the voxels of each DK target region.

**This was run for real in this session** on the entorhinal cortex, ahead of writing this notebook: 1,904
of 1,953 DK left-entorhinal voxels (97.5%) fall inside a single Schaefer-100 parcel,
`7Networks_LH_Limbic_TempPole_1`. The function below is the general form of that same computation, applied
here at Schaefer-200 resolution to every DK target region in the disease list.

**A real constraint this surfaced, not a hypothetical one:** Schaefer is cortex-only. Running this mapping
on Huntington's disease's target (caudate, putamen — both subcortical) returns an empty list, correctly —
there is no Schaefer parcel for the striatum, because Schaefer doesn't parcellate it at all. This isn't a
bug to fix; it's a real fact about the parcellation, and Section 8 reports it as "no Schaefer equivalent"
rather than a missing result. Any disease whose target is entirely subcortical will show the same pattern.

In [ ]:
import nibabel as nib
import numpy as np
from nilearn.image import resample_to_img

dk_img = nib.load(atlas_dk['image'])
dk_data = dk_img.get_fdata()
schaefer_resampled = resample_to_img(schaefer.maps, dk_img, interpolation='nearest', force_resample=True, copy_header=True)
schaefer_data = schaefer_resampled.get_fdata()

def dk_labels_to_schaefer(dk_label_substrings, hemisphere='L', min_overlap_frac=0.15):
    """For each DK region whose label matches dk_label_substrings (a '|'-joined regex,
    same convention as the rest of this notebook), find the Schaefer parcel labels that
    cover at least min_overlap_frac of that DK region's voxels."""
    matched_dk = info_dk[info_dk['label'].str.contains(dk_label_substrings, case=False, regex=True)
                          & (info_dk['hemisphere'] == hemisphere)]
    schaefer_hits = set()
    for _, row in matched_dk.iterrows():
        mask = dk_data == row['id']
        n_voxels = mask.sum()
        if n_voxels == 0:
            continue
        ids, counts = np.unique(schaefer_data[mask], return_counts=True)
        for sid, cnt in zip(ids, counts):
            if sid == 0 or cnt / n_voxels < min_overlap_frac:
                continue
            lbl = sch_labels[int(sid) - 1]
            schaefer_hits.add(lbl)
    return sorted(schaefer_hits)

# sanity check against the value verified above
print(dk_labels_to_schaefer('entorhinal'))

## 5. GWAS Catalog: bulk download, once, filtered per disease

The document's Part 2.3 mentions bulk downloads live at https://www.ebi.ac.uk/gwas/downloads "if you later
want every disease at once" — this is that. Download the full associations file from that page and place it
at `GWAS_DIR/gwas_catalog_full.tsv`. (I'm not hardcoding the exact download filename/URL here: it wasn't
verifiable from this session since `ebi.ac.uk` was blocked, and GWAS Catalog periodically renames dated
release files — check the downloads page directly.)

Filtering is by case-insensitive substring match of each disease's `trait_keyword` against the
`DISEASE/TRAIT` column, combined with the same p<5e-8 genome-wide significance threshold and dash/comma
gene-splitting as the single-disease notebook's Part 2.4. **Print and read the matched trait strings before
trusting a disease's gene list** — this is the direct generalization of the single-disease notebook's own
warning about unverified EFO codes: substring matching can both over-match (an unrelated trait containing
the word) and under-match (a trait phrased differently than the keyword).

In [ ]:
import re

bulk = pd.read_csv(f'{GWAS_DIR}/gwas_catalog_full.tsv', sep='\t', low_memory=False)
bulk = bulk[bulk['P-VALUE'] < 5e-8]

def clean_genes(mapped_gene_series):
    genes = set()
    for entry in mapped_gene_series.dropna().unique():
        for g in re.split(r'[-,]', entry):
            g = g.strip()
            if g:
                genes.add(g)
    return sorted(genes)

gene_lists = {}
match_report = []
for _, d in DISEASES.iterrows():
    subset = bulk[bulk['DISEASE/TRAIT'].str.contains(d['trait_keyword'], case=False, regex=False, na=False)]
    matched_traits = sorted(subset['DISEASE/TRAIT'].unique())
    genes = clean_genes(subset['MAPPED_GENE'])
    gene_lists[d['name']] = genes
    match_report.append({
        'disease': d['name'],
        'n_matched_traits': len(matched_traits),
        'example_traits': matched_traits[:3],
        'n_genes': len(genes),
        'in_range_20_500': 20 <= len(genes) <= 500,
    })

match_report = pd.DataFrame(match_report)
match_report

Inspect `match_report` before continuing. Any disease with `n_matched_traits == 0` matched nothing and
will be skipped below; any with `in_range_20_500 == False` has too few or too many genes for a reliable
permutation test (Part 2.5 of the source document) and is also skipped, with the reason kept in the
report rather than silently dropped — a batch run over 15 diseases should surface these problems, not hide
them.

In [ ]:
usable_diseases = match_report[(match_report['n_matched_traits'] > 0) & (match_report['in_range_20_500'])]['disease'].tolist()
skipped = match_report[~match_report['disease'].isin(usable_diseases)]
print(f"{len(usable_diseases)} of {len(DISEASES)} diseases usable.")
if len(skipped):
    print("Skipped:")
    print(skipped[['disease', 'n_matched_traits', 'n_genes']])

## 6. Run the enrichment for every usable disease, on both parcellations

This is Part 3 of the single-disease notebook (the 10,000-permutation gene-set enrichment test),
generalized into a function and looped. `n_perm` is reduced from the single-disease notebook's 10,000 to
2,000 here — with 15 diseases x 2 parcellations x thousands of permutations x tens of thousands of genes,
10,000 permutations each would be considerably slower in Colab; raise it back to 10,000 for a result you'd
put in front of a reviewer, per the source document's own standard.

In [ ]:
import numpy as np
from statsmodels.stats.multitest import multipletests

def run_enrichment(expression, genes, n_perm=2000, seed=0):
    rng = np.random.default_rng(seed)
    available = [g for g in genes if g in expression.columns]
    if len(available) < 0.5 * len(genes):
        print(f"  WARNING: only {len(available)}/{len(genes)} genes matched")
    observed = expression[available].mean(axis=1)
    all_genes = expression.columns.to_numpy()
    null = np.zeros((n_perm, len(expression)))
    for i in range(n_perm):
        fake = rng.choice(all_genes, size=len(available), replace=False)
        null[i] = expression[fake].mean(axis=1)
    z = (observed - null.mean(axis=0)) / null.std(axis=0)
    p = (null >= observed.to_numpy()).mean(axis=0)
    _, p_fdr, _, _ = multipletests(p, method='fdr_bh')
    return pd.Series(z, index=expression.index), pd.Series(p_fdr, index=expression.index), len(available)

z_grid_dk, z_grid_schaefer = {}, {}
n_matched_report = []
for name in usable_diseases:
    print(f"Running: {name}")
    genes = gene_lists[name]
    z_dk, p_dk, n_matched_dk = run_enrichment(expression_dk, genes)
    z_grid_dk[name] = z_dk
    z_sch, p_sch, n_matched_sch = run_enrichment(expression_schaefer, genes)
    z_grid_schaefer[name] = z_sch
    n_matched_report.append({'disease': name, 'n_genes': len(genes),
                              'n_matched_dk': n_matched_dk, 'n_matched_schaefer': n_matched_sch})

z_grid_dk = pd.DataFrame(z_grid_dk)          # regions x diseases
z_grid_schaefer = pd.DataFrame(z_grid_schaefer)
z_grid_dk.to_csv(f'{SCALE_DIR}/z_grid_dk.csv')
z_grid_schaefer.to_csv(f'{SCALE_DIR}/z_grid_schaefer.csv')
pd.DataFrame(n_matched_report)

## 7. Gene-list-size diagnostic (document's 6.4 requirement, made explicit)

"Diseases with more research behind them have longer gene lists and more statistical power. That
difference will look like biology if you ignore it." This checks directly whether `n_genes` predicts the
strength of the result (mean absolute z-score across regions) — if it does, a chunk of any
cross-disease comparison below (especially the clustering) is gene-list-size, not neurobiology, and
should be reported as such rather than left implicit.

In [ ]:
diag = pd.DataFrame({
    'n_genes': {name: len(gene_lists[name]) for name in usable_diseases},
    'mean_abs_z_dk': z_grid_dk.abs().mean(),
})
corr = diag['n_genes'].corr(diag['mean_abs_z_dk'])
print(f"Correlation between gene-list size and mean |z|: r = {corr:.3f}")
print("If |r| is large, gene-list size is confounding the cross-disease comparison below.")
diag.sort_values('n_genes')

## 8. Target-region check, both parcellations, side by side

For every disease with a `dk_target` defined (Section 1), this reports where that region ranks in each
parcellation's z-score ordering. The Schaefer side uses the verified cross-atlas mapping from Section 4 —
not a second, independent guess at which Schaefer parcel "should" correspond to the DK target.

A result that shows up in both columns replicated across parcellations, satisfying the document's 6.4
requirement directly rather than by assertion. A result in only one column is exactly the kind of
parcellation artifact 6.4 warns about.

In [ ]:
def target_rank(z_series, label_pattern_or_list, info_df, label_col='label', id_col='id'):
    if isinstance(label_pattern_or_list, str):
        matched = info_df[info_df[label_col].str.contains(label_pattern_or_list, case=False, regex=True)]
    else:
        matched = info_df[info_df[label_col].isin(label_pattern_or_list)]
    if matched.empty:
        return None
    ranked = z_series.rank(ascending=False)
    rows = []
    for _, row in matched.iterrows():
        rid = row[id_col]
        if rid in z_series.index:
            rows.append({'label': row[label_col], 'z': z_series[rid], 'rank': int(ranked[rid]), 'n_regions': len(z_series)})
    return rows

target_report = []
for _, d in DISEASES.iterrows():
    if d['name'] not in usable_diseases or pd.isna(d['dk_target']):
        continue
    dk_hits = target_rank(z_grid_dk[d['name']], d['dk_target'], info_dk)
    schaefer_labels = dk_labels_to_schaefer(d['dk_target'])
    sch_hits = target_rank(z_grid_schaefer[d['name']], schaefer_labels, info_schaefer) if schaefer_labels else None
    target_report.append({
        'disease': d['name'],
        'dk_target_pattern': d['dk_target'],
        'dk_best_rank': min((h['rank'] for h in dk_hits), default=None) if dk_hits else None,
        'dk_n_regions': dk_hits[0]['n_regions'] if dk_hits else None,
        'schaefer_target_labels': schaefer_labels,
        'schaefer_best_rank': min((h['rank'] for h in sch_hits), default=None) if sch_hits else None,
        'schaefer_n_regions': sch_hits[0]['n_regions'] if sch_hits else None,
    })

pd.DataFrame(target_report)

## 9. Spatial-null check on the target-region result (document's 6.4: "use spatial nulls throughout")

This applies the single-disease notebook's Part 5.5 spin test to every disease-with-a-target above, on the
DK cortical map, testing each disease's observed z-map against a binary target map for its own target
region — the same logic as before, generalized into a loop instead of hardcoded to Alzheimer's/entorhinal.

**Scope limit, stated plainly:** this is a spatial null for the *target-region* check only. The document's
"use spatial nulls throughout" is not extended here to the clustering step in Section 10 — comparing
*patterns between two disease maps* with a spatial null (rather than one map against a fixed target) is a
different, more involved test (`neuromaps` supports multivariate/pairwise spin comparisons, but that
wasn't verified in this session), and this notebook doesn't claim to have done it. The clustering below is
naive with respect to spatial autocorrelation — treat it as suggestive, not as spin-corrected.

In [ ]:
!pip install -q neuromaps
from neuromaps import nulls
from neuromaps.stats import compare_images

atlas_surf = abagen.fetch_desikan_killiany(surface=True)
parcellation = atlas_surf['image']

spin_report = []
for _, d in DISEASES.iterrows():
    if d['name'] not in usable_diseases or pd.isna(d['dk_target']):
        continue
    cortex_ids = info_dk.loc[(info_dk['structure'] == 'cortex') & (info_dk['hemisphere'] == 'L'), 'id']
    my_map = z_grid_dk[d['name']].loc[z_grid_dk[d['name']].index.isin(cortex_ids)]
    my_map.index = info_dk.set_index('id').loc[my_map.index, 'label']

    target_labels = info_dk[info_dk['label'].str.contains(d['dk_target'], case=False, regex=True)
                             & (info_dk['hemisphere'] == 'L') & (info_dk['structure'] == 'cortex')]['label']
    if target_labels.empty:
        continue  # target is subcortical (e.g. hippocampus) — outside this cortical-surface spin test

    target = pd.Series(0.0, index=my_map.index)
    target.loc[target.index.isin(target_labels)] = 1.0

    rotated = nulls.alexander_bloch(my_map, atlas='fsaverage', density='10k', parcellation=parcellation, n_perm=500)
    corr, spin_p = compare_images(my_map, target, metric='pearsonr', nulls=rotated)
    spin_report.append({'disease': d['name'], 'target': list(target_labels), 'r': corr, 'spin_p': spin_p})

pd.DataFrame(spin_report)

## 10. Clustering: do diseases share a regional pattern?

Hierarchical clustering of diseases by their DK region z-score vectors (document's 6.3: "cluster them by
their region scores. Does the clustering match how doctors group these conditions, or does it cut across
the categories?"). Colour/label by the `category` column from Section 1 (neurodegenerative / psychiatric /
other) so the clinical grouping is visible against the data-driven one.

Per Section 9's scope note: this step is **not** spatial-null-corrected — read the dendrogram as
suggestive of pattern similarity, not as a set of statistically confirmed clusters.

In [ ]:
from scipy.cluster.hierarchy import linkage, dendrogram
import matplotlib.pyplot as plt

cluster_data = z_grid_dk[usable_diseases].T.fillna(0)  # diseases x regions
Z = linkage(cluster_data.values, method='average', metric='correlation')

category_map = DISEASES.set_index('name')['category'].to_dict()
colors = {'neurodegenerative': 'tab:red', 'psychiatric': 'tab:blue', 'other': 'tab:gray'}
labels_with_category = [f"{name} [{category_map[name][:4]}]" for name in cluster_data.index]

fig, ax = plt.subplots(figsize=(8, 6))
dendrogram(Z, labels=labels_with_category, orientation='right', ax=ax)
ax.set_title('Diseases clustered by Desikan-Killiany regional expression pattern')
plt.tight_layout()
plt.show()

## 11. Literature replication — not implemented

The document's third 6.3 question: "take associations that have already been reported in the literature,
redo them with spatial nulls, and count how many hold up. That last one is the strongest angle available to
a beginner."

**This notebook does not implement it.** Doing it honestly requires a curated table of specific
already-published disease-region transcriptomic findings (e.g. from prior AHBA-based papers) to test against
— and fabricating that table, or guessing which findings are "well established" without a citation, would
be exactly the kind of unverified claim this notebook has avoided elsewhere. Two real ways to build that
table:

1. Search the literature yourself (this session has a PubMed connector available) for AHBA/GWAS spatial
   transcriptomic papers per disease, extract their reported region-level findings, and build the ground
   truth table from what you actually read.
2. Use a specific published disease-region atrophy/vulnerability atlas as the comparison target instead of
   a hand-curated list (e.g. ENIGMA consortium case-control cortical thickness maps, where available per
   disease) — this replaces "already reported associations" with an independently measured map, which is
   arguably a stronger test than literature-mining, but is its own separate data-acquisition project.

Either is real work, not a follow-up cell.

## Nearest-gene problem — stated plainly (document's 6.4)

The GWAS Catalog's `MAPPED_GENE` column assigns each significant variant to its nearest gene (or the gene
it falls within), not necessarily the gene that variant actually affects. A variant can be closest to gene
A while its real regulatory effect is on a distant gene B — this is well documented in the GWAS
fine-mapping literature. Every gene list in this notebook, and in the single-disease notebook, inherits this
limitation directly: "risk gene" here means "nearest gene to a significant variant," not "gene demonstrated
to cause the disease." This affects every enrichment result above equally and cannot be fixed by better
statistics on this data alone — it would need eQTL or fine-mapping data GWAS Catalog's basic download
doesn't provide.

## What's left to write up (Part 6 additions to the single-disease notebook's list)

1. Which diseases were skipped and why (Section 5's `skipped` table) — a batch run's failures are data, not
   noise to hide.
2. The gene-list-size correlation from Section 7 — if it's large, say so before presenting the clustering as
   biological.
3. For every disease with a target region: does the DK rank and the Schaefer rank (Section 8) agree? Does
   the spin-corrected p-value (Section 9) still hold after accounting for spatial autocorrelation, or does
   it only look significant in the naive test?
4. Whether the dendrogram (Section 10) separates neurodegenerative from psychiatric conditions, or cuts
   across that boundary — and whether that boundary-crossing (if present) survives the gene-list-size
   caveat from Section 7 rather than being a confound.
5. That literature replication (Section 11) is explicitly unimplemented here, and what it would take.
6. The nearest-gene caveat above — this is a limitation of the GWAS Catalog data itself, not of this
   analysis's execution, same as the six-donor AHBA sample size was in the single-disease notebook.